# Bước 1 — OCR sách Hán bằng PaddleOCR-VL

**Input:** PDF scan trong `/kaggle/input`. **Output:** `runs/ocr_*/txt/<tên sách>.txt` và `ocr_raw/`.

Notebook này chỉ chứa **cấu hình + gọi package**. Toàn bộ logic nằm trong `src/vtm_corpus/`.

**Settings:** Accelerator **GPU T4 x2**, Internet **On**.

## 0. Lấy repo + cài đặt

In [ ]:
# ==== Repo / runtime ====
# Repo công khai: điền URL sau khi publish.
REPO_URL = "https://github.com/nthday-jpg/chinese-vietnamese-vtm-corpus.git"              # vd "https://github.com/you/vtm-corpus.git"
REPO_DIR = "/kaggle/working/vtm-corpus"
INPUT_DIR = "/kaggle/input/datasets/nthday/vietnamese-traditional-medicine"   # thư mục chứa PDF (bước 1) hoặc .txt (bước 2)

import subprocess, sys
from pathlib import Path

def sh(cmd, check=True):
    print(">", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.stdout.strip(): print(r.stdout.strip())
    if r.stderr.strip(): print(r.stderr.strip())
    if check and r.returncode:
        raise SystemExit(r.returncode)
    return r

local_src = Path.cwd() / "src"
if (local_src / "vtm_corpus").exists():
    REPO_DIR = str(Path.cwd())
    sys.path.insert(0, str(local_src))
elif REPO_URL:
    if not (Path(REPO_DIR) / ".git").exists():
        sh(f"git clone --quiet {REPO_URL} {REPO_DIR}")
    else:
        sh(f"git -C {REPO_DIR} pull --quiet")
    sys.path.insert(0, str(Path(REPO_DIR) / "src"))
else:
    try:
        import vtm_corpus
    except ImportError:
        raise SystemExit("Chưa có repo: điền REPO_URL hoặc chạy notebook trong repo đã clone.")
print("REPO_DIR =", REPO_DIR)

In [ ]:
# Cài package vào Python hệ thống + PaddleOCR-VL vào venv riêng.
sh(f'{sys.executable} -m pip install --upgrade uv')
sh(f'{sys.executable} -m uv --version')
sh(f'{sys.executable} -m pip install -q -e "{REPO_DIR}[swinir]" pypdfium2')
try:  # model cần đăng nhập: Add-ons -> Secrets -> HF_TOKEN
    from kaggle_secrets import UserSecretsClient
    import os
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
VENV = "/tmp/venv_paddle"   # venv riêng: Paddle dùng CUDA library khác torch của Kaggle
PY_PADDLE = f"{VENV}/bin/python"
if not Path(PY_PADDLE).exists():
    sh(f'{sys.executable} -m uv venv --python 3.11 {VENV}')
sh(f'{sys.executable} -m uv pip install --quiet --python "{PY_PADDLE}" "paddlepaddle-gpu==3.2.1" '
   f'--index https://www.paddlepaddle.org.cn/packages/stable/cu126/ '
   f'--index https://pypi.org/simple --index-strategy unsafe-best-match')
sh(f'{sys.executable} -m uv pip install --quiet --python "{PY_PADDLE}" '
   f'"paddleocr[doc-parser]" pypdfium2 pillow')
sh(f'{PY_PADDLE} -c "import paddle; paddle.utils.run_check()"')
print("Đã cài xong. Worker Python:", PY_PADDLE)

## 1. Xem danh sách đầu vào (PDF hoặc txt)


In [ ]:
# ==== XEM DANH SÁCH PDF ĐẦU VÀO ====
# Chạy ô này trước để biết tên sách, rồi điền `books` trong ô cấu hình bên dưới.
#   books=[]                         -> tất cả sách
#   books=["Vệ_sinh_yếu_chỉ_-_tập_4"] -> đúng 1 sách
#   books=["*ky_huyet*", "*nhi*"]     -> nhiều sách theo glob
from vtm_corpus import list_pdfs

for i, e in enumerate(list_pdfs(INPUT_DIR), 1):
    print(f"{i:3}. {e['name']:<50} {e['n_pages']:>4} trang  {e['path']}")

## 2. Cấu hình (sửa mọi tham số tại đây)


In [ ]:
# ==== CẤU HÌNH OCR — sửa mọi trường ngay tại đây ====
from vtm_corpus import OCRConfig, ReportConfig
from vtm_corpus.config import LAYOUT_PROMPT

OCR = OCRConfig(
    engine="paddle",                  # hunyuan | paddle | dummy
    input_dir=INPUT_DIR,              # thư mục chứa PDF (glob đệ quy)
    books=[],                         # [] = tất cả; tên/glob, vd ["*ky_huyet*"]
    workdir="/kaggle/working",        # nơi tạo runs/
    run_name="paddle",                # "" = dùng engine
    pages="",                         # "" = cả sách; "1-10,15" = chạy thử
    dpi=200,
    max_side=1600,
    swinir=True,                      # False = bỏ khử nhiễu; không upscale
    swinir_cache="/kaggle/working/preprocessed",  # dùng chung giữa các engine
    max_new_tokens=6000,
    dtype="fp32",                     # fp32 | fp16 | bf16
    model="PaddleOCR-VL",             # không dùng cho paddle; giữ để đủ trường
    prompt="",                        # "" = prompt mặc định
    repetition_penalty=1.08,
    device="auto",                    # auto | cpu | cuda | cuda:1
    paddle_version="v1.6",
    paddle_args={},
    limit=None,                       # vd 2 = chỉ chạy 2 trang để thử
    page_timeout=300,
    n_gpu=0,                          # 0 = tự phát hiện
    worker_python=PY_PADDLE,          # worker Paddle chạy trong venv riêng
    budget_h=10.5,
    stall_min=15,
    max_restarts=5,
    n_layout=3,                       # 0 = tắt layout
    layout_budget_h=1.0,
    layout_prompt=LAYOUT_PROMPT,
    export=True,
    fresh=False,                      # True = xoá run cũ cùng cấu hình
    resume_from=[],                   # vd ["/kaggle/input/<notebook-output>"]
)

REPORT = ReportConfig(
    preview_pages=3,
    preview_chars=1500,
    layout_images=6,
    show_tables=True,
    image_width=700,
)

## 3. Tiền xử lý riêng, rồi chạy OCR

SwinIR chạy bằng Python hệ thống (PyTorch), tự dùng CUDA nếu có và lưu PNG. Tiến trình này kết thúc trước OCR; worker Paddle trong venv riêng chỉ đọc cache. Chạy lại sẽ bỏ qua trang đã lưu. Đặt `swinir=False` để bỏ qua; nên thử `pages="1-20"` trước khi chạy toàn bộ.


In [ ]:
# ==== CHẠY OCR ====
# Dùng Python hệ thống cho SwinIR, không dùng PY_PADDLE.
import json, tempfile
if OCR.swinir:
    with tempfile.TemporaryDirectory() as tmp:
        config = Path(tmp) / "preprocess.json"
        config.write_text(json.dumps(OCR.to_dict(), ensure_ascii=False), encoding="utf-8")
        subprocess.run([sys.executable, "-m", "vtm_corpus.cli", "preprocess", "--config", str(config)], check=True)

from vtm_corpus import run_ocr

run = run_ocr(OCR)
print("run_dir:   ", run.run_dir)
print("output_dir:", run.output_dir)
run

## 4. Xem kết quả ngay trong notebook


In [ ]:
# ==== XEM KẾT QUẢ NGAY TRONG NOTEBOOK ====
from vtm_corpus import show_layouts, show_ocr_report, show_pages

show_ocr_report(run.run_dir, REPORT)
show_layouts(run.run_dir, REPORT)
show_pages(run.run_dir, REPORT)

## 5. Chạy thử nhanh trước khi chạy full


In [ ]:
# ==== CHẠY THỬ NHANH (tuỳ chọn) ====
# Bỏ comment để chạy 2 trang đầu, không vẽ layout, rồi xem kết quả.
# OCR.pages = "1-2"
# OCR.limit = 2
# OCR.n_layout = 0
# run = run_ocr(OCR)
# show_ocr_report(run.run_dir, REPORT)
# show_pages(run.run_dir, REPORT)